# Finetune Baseline and Foundation Model Comparison


In [1]:

import json
import logging
import random
import time
import traceback
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
)
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)

PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_DISPLAY = ['NORM', 'AMI', 'IMI', 'LMI']
MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
LEAD_ORDER = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

class PerLeadNormalizer:
    def __init__(self, eps=1e-6):
        self.eps=eps; self.mean=None; self.std=None
    def fit(self,x):
        self.mean=x.mean(axis=(0,1),keepdims=True); self.std=np.maximum(x.std(axis=(0,1),keepdims=True),self.eps); return self
    def transform(self,x): return ((x-self.mean)/self.std).astype(np.float32)
    def save(self,path):
        np.save(Path(path)/'normalizer_mean.npy', self.mean.astype(np.float32)); np.save(Path(path)/'normalizer_std.npy', self.std.astype(np.float32))
    def load(self,path):
        self.mean=np.load(Path(path)/'normalizer_mean.npy'); self.std=np.load(Path(path)/'normalizer_std.npy'); return self

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def setup_logger(name, log_file, error_file=None, reset=False):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if reset:
        for handler in list(logger.handlers):
            logger.removeHandler(handler)
            handler.close()
    formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    stream_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    if error_file is not None:
        error_file = Path(error_file)
        error_file.parent.mkdir(parents=True, exist_ok=True)
        error_handler = logging.FileHandler(error_file)
        error_handler.setLevel(logging.ERROR)
        error_handler.setFormatter(formatter)
        logger.addHandler(error_handler)
    return logger

def parse_labels(labels_df, label_arr):
    if 'main_label_name' in labels_df.columns:
        return labels_df['main_label_name'].map(MAIN_LABEL_TO_INDEX).to_numpy(np.int64)
    if label_arr.ndim > 1:
        return label_arr.argmax(axis=1).astype(np.int64)
    return label_arr.reshape(-1).astype(np.int64)

def load_split(dataset_dir, split):
    split_dir=Path(dataset_dir)/split
    x=np.load(split_dir/'x_beats.npy').astype(np.float32)
    label_arr=np.load(split_dir/'main_label.npy')
    meta=pd.read_csv(split_dir/'metadata.csv')
    y=parse_labels(meta,label_arr)
    return x,y,meta

class ECGDataset(Dataset):
    def __init__(self,x,y,indices=None):
        self.x=torch.tensor(x,dtype=torch.float32); self.y=torch.tensor(y,dtype=torch.long)
        self.indices=np.arange(len(y)) if indices is None else np.asarray(indices)
    def __len__(self): return len(self.y)
    def __getitem__(self,idx): return {'x':self.x[idx], 'y':self.y[idx], 'idx':torch.tensor(int(self.indices[idx]),dtype=torch.long)}

def softmax_np(logits):
    logits=logits-logits.max(axis=1,keepdims=True); exp=np.exp(logits); return exp/exp.sum(axis=1,keepdims=True)

def compute_metrics(y_true, logits):
    prob=softmax_np(logits); pred=prob.argmax(axis=1); cm=confusion_matrix(y_true,pred,labels=np.arange(len(MAIN_LABELS)))
    out={'accuracy':float(accuracy_score(y_true,pred)),'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),'macro_f1':float(f1_score(y_true,pred,average='macro',zero_division=0)),'micro_f1':float(f1_score(y_true,pred,average='micro',zero_division=0)),'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0))}
    per=f1_score(y_true,pred,labels=np.arange(len(MAIN_LABELS)),average=None,zero_division=0); class_rows=[]; aucs=[]
    for idx,(label,display,val) in enumerate(zip(MAIN_LABELS,MAIN_LABEL_DISPLAY,per)):
        tp=float(cm[idx,idx]); fn=float(cm[idx,:].sum()-cm[idx,idx]); fp=float(cm[:,idx].sum()-cm[idx,idx]); tn=float(cm.sum()-tp-fn-fp)
        sens=tp/max(tp+fn,1.0); spec=tn/max(tn+fp,1.0)
        try: auc=float(roc_auc_score((y_true==idx).astype(int),prob[:,idx])); aucs.append(auc)
        except ValueError: auc=np.nan
        out[f'f1_{display}']=float(val); out[f'sensitivity_{display}']=float(sens); out[f'specificity_{display}']=float(spec); out[f'auc_{display}']=auc
        class_rows.append({'label':display,'internal_label':label,'f1_score':float(val),'sensitivity':float(sens),'specificity':float(spec),'auc':auc,'support':int((y_true==idx).sum())})
    out['macro_auc']=float(np.nanmean(aucs)) if aucs else np.nan
    return out, prob, pred, pd.DataFrame(class_rows)

def save_curves(y_true, prob, metrics_dir, prefix):
    metrics_dir=Path(metrics_dir); metrics_dir.mkdir(parents=True,exist_ok=True)
    roc_rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max(): continue
        fpr,tpr,_=roc_curve(binary,prob[:,idx]); auc=roc_auc_score(binary,prob[:,idx])
        ax.plot(fpr,tpr,linewidth=2.2,label=f'{label} AUC={auc:.3f}')
        roc_rows.extend([{'label':label,'fpr':float(x),'tpr':float(y),'auc':float(auc)} for x,y in zip(fpr,tpr)])
    ax.plot([0,1],[0,1],'--',color='black'); ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate / Sensitivity'); ax.legend(); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_roc_curve.png',bbox_inches='tight'); plt.close(fig)
    pd.DataFrame(roc_rows).to_csv(metrics_dir/f'{prefix}_roc_curve.csv',index=False)
    pr_rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max(): continue
        precision,recall,_=precision_recall_curve(binary,prob[:,idx]); auprc=average_precision_score(binary,prob[:,idx])
        ax.plot(recall,precision,linewidth=2.2,label=f'{label} AUPRC={auprc:.3f}')
        pr_rows.extend([{'label':label,'recall':float(r),'precision':float(p),'auprc':float(auprc)} for r,p in zip(recall,precision)])
    ax.set_xlabel('Recall / Sensitivity'); ax.set_ylabel('Precision'); ax.legend(); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_pr_curve.png',bbox_inches='tight'); plt.close(fig)
    pd.DataFrame(pr_rows).to_csv(metrics_dir/f'{prefix}_pr_curve.csv',index=False)

def save_confusion(y_true,pred,metrics_dir,prefix):
    cm=confusion_matrix(y_true,pred,labels=np.arange(len(MAIN_LABELS))); metrics_dir=Path(metrics_dir)
    pd.DataFrame(cm,index=MAIN_LABEL_DISPLAY,columns=MAIN_LABEL_DISPLAY).to_csv(metrics_dir/f'{prefix}_confusion_matrix.csv')
    fig,ax=plt.subplots(figsize=(9,8),dpi=400); sns.heatmap(cm,annot=True,fmt='d',cmap='Oranges',xticklabels=MAIN_LABEL_DISPLAY,yticklabels=MAIN_LABEL_DISPLAY,linewidths=1,linecolor='black',ax=ax,annot_kws={'fontsize':14})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_confusion_matrix.png',bbox_inches='tight'); plt.close(fig)

def plot_training(rows,metrics_dir):
    df=pd.DataFrame(rows); df.to_csv(Path(metrics_dir)/'metrics.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(14,5),dpi=300)
    axes[0].plot(df['epoch'],df['train_loss'],label='train'); axes[0].plot(df['epoch'],df['val_loss'],label='val'); axes[0].legend(); axes[0].grid(True,alpha=.3); axes[0].set_title('Loss')
    axes[1].plot(df['epoch'],df['train_macro_f1'],label='train'); axes[1].plot(df['epoch'],df['val_macro_f1'],label='val'); axes[1].legend(); axes[1].grid(True,alpha=.3); axes[1].set_title('Macro F1')
    fig.tight_layout(); fig.savefig(Path(metrics_dir)/'training_curve.png',bbox_inches='tight'); plt.close(fig)

def run_epoch(model,loader,criterion,optimizer=None,device='cpu',max_batches=None):
    training=optimizer is not None; model.train(training); total=0; n=0; logits=[]; y=[]; idx=[]
    ctx=torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi,b in enumerate(loader):
            if max_batches and bi>=max_batches: break
            xb=b['x'].to(device); yb=b['y'].to(device)
            if training: optimizer.zero_grad(set_to_none=True)
            out=model(xb); loss=criterion(out,yb)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            total += float(loss.detach().cpu())*xb.size(0); n += xb.size(0)
            logits.append(out.detach().cpu().numpy()); y.append(yb.detach().cpu().numpy()); idx.append(b['idx'].detach().cpu().numpy())
    return {'loss':total/max(n,1),'logits':np.concatenate(logits),'y_true':np.concatenate(y),'idx':np.concatenate(idx)}

class CNN1D(nn.Module):
    def __init__(self,input_leads=12,num_classes=4):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(input_leads,64,7,padding=3),nn.BatchNorm1d(64),nn.GELU(),nn.MaxPool1d(2),nn.Conv1d(64,128,5,padding=2),nn.BatchNorm1d(128),nn.GELU(),nn.AdaptiveAvgPool1d(1)); self.head=nn.Linear(128,num_classes)
    def forward(self,x): x=x.permute(0,2,1); return self.head(self.net(x).squeeze(-1))
class LSTMModel(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.rnn=nn.LSTM(input_leads,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): out,_=self.rnn(x); return self.head(out[:,-1])
class GRUModel(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.rnn=nn.GRU(input_leads,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): out,_=self.rnn(x); return self.head(out[:,-1])
class CNNLSTM(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.cnn=nn.Sequential(nn.Conv1d(input_leads,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.Conv1d(64,64,3,padding=1),nn.BatchNorm1d(64),nn.GELU()); self.rnn=nn.LSTM(64,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): z=self.cnn(x.permute(0,2,1)).permute(0,2,1); out,_=self.rnn(z); return self.head(out[:,-1])

def build_model(name):
    if name=='cnn1d': return CNN1D()
    if name=='lstm': return LSTMModel()
    if name=='gru': return GRUModel()
    if name=='cnn_lstm': return CNNLSTM()
    raise ValueError(name)

CONFIG={'seed':42,'dataset_dir':str(PROJECT_ROOT/'dataset'/'ptb_diagnostic'),'pretrain_root':str(PROJECT_ROOT/'outputs'/'6_model_pretrain_comparison'),'pretrain_run_id':'latest','output_base_dir':str(PROJECT_ROOT/'outputs'/'7_model_finetune_comparison'),'batch_size':64,'epochs':50,'learning_rate':1e-4,'external_learning_rate':1e-5,'weight_decay':1e-4,'device':'cuda' if torch.cuda.is_available() else 'cpu','models':['cnn1d','lstm','gru','cnn_lstm','hubert_ecg','ecg_fm'],'hubert_ecg_model_id_or_path':str(PROJECT_ROOT.parent/'myocardial-infarction-classification'/'model_pretrain_comparison'/'hubert_ecg'),'ecg_fm_checkpoint_path':str(PROJECT_ROOT.parent/'myocardial-infarction-classification'/'model_pretrain_comparison'/'ecg_fm'/'mimic_iv_ecg_physionet_pretrained.pt')}
set_seed(CONFIG['seed']); OUTPUT_DIR=Path(CONFIG['output_base_dir'])/RUN_TIMESTAMP; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
GLOBAL_LOG_DIR = OUTPUT_DIR / 'logs'
LOGGER = setup_logger('model_finetune_comparison', GLOBAL_LOG_DIR/'train.log', GLOBAL_LOG_DIR/'error.log', reset=True)
run_started_at = time.time()
LOGGER.info('Model finetune comparison logger initialized.')
LOGGER.info('Run directory: %s', OUTPUT_DIR)
LOGGER.info('Device: %s', CONFIG['device'])
LOGGER.info('Models: %s', ', '.join(CONFIG['models']))
LOGGER.info('Epochs: %s | Batch size: %s | Learning rate: %s', CONFIG['epochs'], CONFIG['batch_size'], CONFIG['learning_rate'])
with open(OUTPUT_DIR/'config.json','w') as f: json.dump(CONFIG,f,indent=2)
LOGGER.info('Configuration saved: %s', OUTPUT_DIR/'config.json')

def latest_pretrain(root):
    root=Path(root); runs=sorted([p for p in root.iterdir() if p.is_dir()])
    if not runs: raise FileNotFoundError(root)
    return runs[-1]
PRETRAIN_RUN=latest_pretrain(CONFIG['pretrain_root']) if CONFIG['pretrain_run_id']=='latest' else Path(CONFIG['pretrain_root'])/CONFIG['pretrain_run_id']
LOGGER.info('Pretrain source: %s', PRETRAIN_RUN)

x_train,y_train,meta_train=load_split(CONFIG['dataset_dir'],'train'); x_val,y_val,meta_val=load_split(CONFIG['dataset_dir'],'val'); x_test,y_test,meta_test=load_split(CONFIG['dataset_dir'],'test')
LOGGER.info('Dataset loaded from %s', CONFIG['dataset_dir'])
LOGGER.info('Train shape=%s | Val shape=%s | Test shape=%s', x_train.shape, x_val.shape, x_test.shape)
LOGGER.info('Train label counts=%s', np.bincount(y_train, minlength=len(MAIN_LABELS)).tolist())
LOGGER.info('Val label counts=%s', np.bincount(y_val, minlength=len(MAIN_LABELS)).tolist())
LOGGER.info('Test label counts=%s', np.bincount(y_test, minlength=len(MAIN_LABELS)).tolist())
summary=[]
EXTERNAL_MODEL_TARGET_LENGTH = 1000

def resample_external_ecg(x, target_length=EXTERNAL_MODEL_TARGET_LENGTH):
    if x.ndim != 3:
        raise ValueError(f'Expected ECG tensor [B,L,12], got shape={tuple(x.shape)}')
    if x.shape[1] == target_length:
        return x
    x_bcl = x.permute(0,2,1).contiguous()
    x_bcl = F.interpolate(x_bcl, size=target_length, mode='linear', align_corners=False)
    return x_bcl.permute(0,2,1).contiguous()

def materialize_lazy_model(model, sample_x, device):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        _ = model(sample_x[:1].to(device))
    model.train(was_training)

class HuBERT_ECG_Classifier(nn.Module):
    def __init__(self, model_id, num_classes=4, dropout=0.3, target_length=EXTERNAL_MODEL_TARGET_LENGTH):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(model_id, trust_remote_code=True)
        hidden_size = int(self.backbone.config.hidden_size)
        self.target_length = target_length
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        # Beat inputs are [B,65,12]; HuBERT ECG expects a longer raw waveform.
        x = resample_external_ecg(x, self.target_length)
        x_mean = x.mean(dim=2)
        out = self.backbone(input_values=x_mean)
        pooled = out.last_hidden_state.mean(dim=1)
        return self.fc(self.dropout(pooled))

class ECGFMClassifier(nn.Module):
    def __init__(self, checkpoint_path, num_classes=4, dropout=0.3, pooling='mean', target_length=EXTERNAL_MODEL_TARGET_LENGTH):
        super().__init__()
        from fairseq_signals.models import build_model_from_checkpoint
        self.backbone = build_model_from_checkpoint(checkpoint_path=str(checkpoint_path))
        self.pooling = pooling
        self.target_length = target_length
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.LazyLinear(num_classes)
    def forward(self, x):
        x = resample_external_ecg(x, self.target_length)
        x_bcl = x.permute(0,2,1).contiguous()
        out = self.backbone(source=x_bcl)
        features = out.get('features') if isinstance(out, dict) else None
        if features is None:
            raise TypeError('ECG-FM output does not contain features tensor')
        pooled = features.max(dim=1).values if self.pooling == 'max' else features.mean(dim=1)
        return self.fc(self.dropout(pooled))

def maybe_external_model(name):
    if name == 'hubert_ecg':
        model_id = CONFIG.get('hubert_ecg_model_id_or_path')
        if not model_id or not Path(model_id).exists():
            return None, f'missing HuBERT ECG model path: {model_id}'
        try:
            return HuBERT_ECG_Classifier(model_id, num_classes=len(MAIN_LABELS)), f'loaded HuBERT ECG from {model_id}'
        except Exception as exc:
            return None, f'failed loading HuBERT ECG: {repr(exc)}'
    if name == 'ecg_fm':
        checkpoint_path = Path(CONFIG.get('ecg_fm_checkpoint_path',''))
        if not checkpoint_path.exists():
            return None, f'missing ECG-FM checkpoint: {checkpoint_path}'
        try:
            return ECGFMClassifier(checkpoint_path, num_classes=len(MAIN_LABELS)), f'loaded ECG-FM from {checkpoint_path}'
        except Exception as exc:
            return None, f'failed loading ECG-FM: {repr(exc)}'
    return None, f'unknown external model: {name}'

for model_name in CONFIG['models']:
    model_dir=OUTPUT_DIR/model_name; metrics_dir=model_dir/'metrics'; ckpt_dir=model_dir/'checkpoints'; pred_dir=model_dir/'predictions'; model_log_dir=model_dir/'logs'
    for d in [metrics_dir,ckpt_dir,pred_dir,model_dir/'configs',model_log_dir]: d.mkdir(parents=True,exist_ok=True)
    model_logger = setup_logger(f'model_finetune_comparison.{model_name}', model_log_dir/'train.log', model_log_dir/'error.log', reset=True)
    with open(model_dir/'configs'/'config.json','w') as f: json.dump({**CONFIG,'model_name':model_name},f,indent=2)
    LOGGER.info('Model started: %s', model_name)
    model_logger.info('Model started: %s', model_name)
    model_started_at = time.time()
    try:
        if model_name in {'hubert_ecg','ecg_fm'}:
            normalizer=PerLeadNormalizer().fit(x_train)
            model, reason = maybe_external_model(model_name)
            if model is None:
                LOGGER.warning('Model skipped: %s | %s', model_name, reason)
                model_logger.warning('Model skipped: %s', reason)
                summary.append({'model_name':model_name,'status':'SKIPPED','reason':reason,'output_dir':str(model_dir)})
                continue
        else:
            transfer=PRETRAIN_RUN/model_name/'transfer_ready'; normalizer=PerLeadNormalizer().load(transfer)
            model=build_model(model_name)
            ckpt_path=transfer/f'{model_name}_full_model.pt'
            payload=torch.load(ckpt_path,map_location='cpu')
            model.load_state_dict(payload['model_state_dict'],strict=True)
            reason='loaded local pretrained checkpoint'
        LOGGER.info('%s load status: %s', model_name, reason)
        model_logger.info('Load status: %s', reason)
        xtr=normalizer.transform(x_train); xva=normalizer.transform(x_val); xte=normalizer.transform(x_test)
        train_loader=DataLoader(ECGDataset(xtr,y_train),batch_size=CONFIG['batch_size'],shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
        val_loader=DataLoader(ECGDataset(xva,y_val),batch_size=CONFIG['batch_size'],shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available())
        test_loader=DataLoader(ECGDataset(xte,y_test),batch_size=CONFIG['batch_size'],shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available())
        model=model.to(CONFIG['device'])
        if model_name in {'hubert_ecg','ecg_fm'}:
            dummy_x = torch.as_tensor(xtr[:1], dtype=torch.float32, device=CONFIG['device'])
            materialize_lazy_model(model, dummy_x, CONFIG['device'])
            LOGGER.info('%s external forward probe passed with input shape=%s target_length=%s', model_name, tuple(dummy_x.shape), EXTERNAL_MODEL_TARGET_LENGTH)
            model_logger.info('External forward probe passed with input shape=%s target_length=%s', tuple(dummy_x.shape), EXTERNAL_MODEL_TARGET_LENGTH)
        total_params=sum(p.numel() for p in model.parameters())
        trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
        LOGGER.info('%s parameters trainable=%s total=%s ratio=%.4f', model_name, trainable_params, total_params, trainable_params/max(total_params,1))
        model_logger.info('Parameters trainable=%s total=%s ratio=%.4f', trainable_params, total_params, trainable_params/max(total_params,1))
        criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG['learning_rate'],weight_decay=CONFIG['weight_decay']); scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=5)
        best=-np.inf; rows=[]
        for epoch in range(1,CONFIG['epochs']+1):
            tr=run_epoch(model,train_loader,criterion,optimizer,CONFIG['device']); va=run_epoch(model,val_loader,criterion,None,CONFIG['device'])
            tm,_,_,_=compute_metrics(tr['y_true'],tr['logits']); vm,_,_,_=compute_metrics(va['y_true'],va['logits']); scheduler.step(vm['macro_f1'])
            rows.append({'epoch':epoch,'train_loss':tr['loss'],'val_loss':va['loss'],'train_macro_f1':tm['macro_f1'],'val_macro_f1':vm['macro_f1'],'learning_rate':optimizer.param_groups[0]['lr']})
            epoch_message = (f"Epoch {epoch:03d}/{CONFIG['epochs']:03d} train_loss={tr['loss']:.5f} val_loss={va['loss']:.5f} "
                             f"train_macro_f1={tm['macro_f1']:.4f} val_macro_f1={vm['macro_f1']:.4f} lr={optimizer.param_groups[0]['lr']:.6g}")
            LOGGER.info('%s | %s', model_name, epoch_message)
            model_logger.info(epoch_message)
            torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'config':CONFIG,'model_name':model_name},ckpt_dir/'last.pt')
            if vm['macro_f1']>best:
                best=vm['macro_f1']; torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'best_macro_f1':best,'config':CONFIG,'model_name':model_name},ckpt_dir/'best_macro_f1.pt')
                LOGGER.info('%s new best val_macro_f1=%.4f at epoch %s', model_name, best, epoch)
                model_logger.info('New best val_macro_f1=%.4f at epoch %s', best, epoch)
        plot_training(rows,metrics_dir)
        payload=torch.load(ckpt_dir/'best_macro_f1.pt',map_location=CONFIG['device']); model.load_state_dict(payload['model_state_dict'])
        val=run_epoch(model,val_loader,criterion,None,CONFIG['device']); test=run_epoch(model,test_loader,criterion,None,CONFIG['device'])
        vm,vp,vpred,vclass=compute_metrics(val['y_true'],val['logits']); tm,tp,tpred,tclass=compute_metrics(test['y_true'],test['logits'])
        pd.DataFrame([{'split':'val',**vm,'loss':val['loss']},{'split':'test',**tm,'loss':test['loss']}]).to_csv(metrics_dir/'final_metrics.csv',index=False)
        vclass.insert(0,'split','val'); tclass.insert(0,'split','test'); pd.concat([vclass,tclass],ignore_index=True).to_csv(metrics_dir/'per_class_metrics.csv',index=False)
        save_confusion(val['y_true'],vpred,metrics_dir,'val'); save_confusion(test['y_true'],tpred,metrics_dir,'test'); save_curves(val['y_true'],vp,metrics_dir,'val'); save_curves(test['y_true'],tp,metrics_dir,'test')
        df=meta_test.iloc[test['idx']].copy().reset_index(drop=True); df['true_class_index']=test['y_true']; df['pred_class_index']=tpred
        for i,n in enumerate(MAIN_LABELS): df[f'prob_{n}']=tp[:,i]
        df.to_csv(pred_dir/'test_predictions.csv',index=False)
        elapsed = time.time() - model_started_at
        LOGGER.info('Model finished: %s | test_macro_f1=%.4f | test_loss=%.5f | elapsed=%.1fs', model_name, tm.get('macro_f1', float('nan')), test['loss'], elapsed)
        model_logger.info('Model finished | test_macro_f1=%.4f | test_loss=%.5f | elapsed=%.1fs', tm.get('macro_f1', float('nan')), test['loss'], elapsed)
        summary.append({'model_name':model_name,'status':'OK','reason':reason,**{f'test_{k}':v for k,v in tm.items() if isinstance(v,(float,int,np.floating))},'output_dir':str(model_dir),'checkpoint_path':str(ckpt_dir/'best_macro_f1.pt')})
    except Exception as exc:
        LOGGER.error('Model failed: %s | %s', model_name, exc, exc_info=True)
        model_logger.error('Model failed: %s', exc, exc_info=True)
        traceback.print_exc()
        summary.append({'model_name':model_name,'status':'FAILED','reason':str(exc),'output_dir':str(model_dir)})
summary_df=pd.DataFrame(summary); summary_df.to_csv(OUTPUT_DIR/'metrics_summary.csv',index=False)
LOGGER.info('Summary saved: %s', OUTPUT_DIR/'metrics_summary.csv')
LOGGER.info('Run finished in %.1fs', time.time() - run_started_at)
display(summary_df)


2026-06-11 20:47:40,690 | INFO | Model finetune comparison logger initialized.


2026-06-11 20:47:40,691 | INFO | Run directory: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/7_model_finetune_comparison/20260611_204740


2026-06-11 20:47:40,691 | INFO | Device: cuda


2026-06-11 20:47:40,691 | INFO | Models: cnn1d, lstm, gru, cnn_lstm, hubert_ecg, ecg_fm


2026-06-11 20:47:40,692 | INFO | Epochs: 50 | Batch size: 64 | Learning rate: 0.0001


...

2026-06-11 21:03:44,302 | INFO | Model finished: ecg_fm | test_macro_f1=0.1347 | test_loss=1.61981 | elapsed=646.5s


2026-06-11 21:03:44,302 | INFO | Model finished | test_macro_f1=0.1347 | test_loss=1.61981 | elapsed=646.5s


2026-06-11 21:03:44,304 | INFO | Summary saved: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/7_model_finetune_comparison/20260611_204740/metrics_summary.csv


2026-06-11 21:03:44,304 | INFO | Run finished in 963.6s


,model_name,status,reason,test_accuracy,test_balanced_accuracy,test_macro_f1,test_micro_f1,test_weighted_f1,test_f1_NORM,test_sensitivity_NORM,...,test_sensitivity_IMI,test_specificity_IMI,test_auc_IMI,test_f1_LMI,test_sensitivity_LMI,test_specificity_LMI,test_auc_LMI,test_macro_auc,output_dir,checkpoint_path
0,cnn1d,OK,loaded local pretrained checkpoint,0.637275,0.583423,0.540924,0.637275,0.595671,0.614286,0.955556,...,0.635870,0.946032,0.875069,0.000000,0.000000,1.00000,0.913172,0.912431,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
1,lstm,OK,loaded local pretrained checkpoint,0.709419,0.665761,0.607699,0.709419,0.661692,0.658537,1.000000,...,0.663043,1.000000,0.926242,0.000000,0.000000,1.00000,0.432634,0.821916,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
2,gru,OK,loaded local pretrained checkpoint,0.729459,0.667394,0.607931,0.729459,0.673957,0.705570,0.985185,...,0.777174,0.965079,0.951346,0.000000,0.000000,1.00000,0.838537,0.908111,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
3,cnn_lstm,OK,loaded local pretrained checkpoint,0.873747,0.879471,0.875153,0.873747,0.873802,0.858108,0.940741,...,0.831522,0.965079,0.921498,0.902857,0.951807,0.96875,0.993252,0.959863,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
4,hubert_ecg,OK,loaded HuBERT ECG from /home/nugee/code-progra...,0.693387,0.624544,0.592181,0.693387,0.639473,0.827839,0.837037,...,0.826087,0.615873,0.729658,0.000000,0.000000,1.00000,0.933793,0.898439,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
5,ecg_fm,OK,loaded ECG-FM from /home/nugee/code-program/co...,0.368737,0.250000,0.134700,0.368737,0.198676,0.000000,0.000000,...,1.000000,0.000000,0.519427,0.000000,0.000000,1.00000,0.509543,0.589215,/home/nugee/code-program/code-thesis/hibah/myo...,/home/nugee/code-program/code-thesis/hibah/myo...
